<a href="https://colab.research.google.com/github/syedshadab14/NLP_PROJECTS/blob/main/NLP_fake_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORTING LIBRERIES

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df0=pd.read_csv('/content/Fake.csv')
df0.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [ ]:
df1=pd.read_csv('/content/True.csv')
df1.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


#df0 =fake data set
#df1 =true data set

In [ ]:
df0['label']=0
df1['label']=1


In [ ]:
#concating both data set
df=pd.concat([df0,df1],axis=0)

In [ ]:
#shuffeling
df=df.sample(frac=1).reset_index(drop=True)

In [ ]:
df.head()

,title,text,subject,date,label
0,Clinton’s Server Employee On Hillary’s E-mails...,As Hillary does her best to deflect and stonew...,politics,"Oct 7, 2015",0
1,"Purge of Saudi princes, businessmen widens, tr...",RIYADH (Reuters) - A campaign of mass arrests ...,worldnews,"November 6, 2017",1
2,Here’s A LONG List Of Bernie Sanders’ Accompl...,We ve already discussed Barack Obama s many ac...,News,"February 19, 2016",0
3,KARMA: GAY PASTOR SUES WHOLE FOODS For “Anti-G...,Too bad for the gay pastor that Whole Foods ha...,politics,"Apr 20, 2016",0
4,BREAKING BAD: John McCain’s Campaign Rocked by...,21st Century Wire asks Will this be the beginn...,Middle-east,"April 28, 2016",0


**intial inspection**

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   label    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB


In [ ]:
df.describe()

,label
count,44898.000000
mean,0.477015
std,0.499477
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


In [ ]:
df.shape

(44898, 5)

# **checking null value**

In [ ]:
df.isna().sum()

,0
title,0
text,0
subject,0
date,0
label,0


cleaning

In [ ]:
#function
import re
def clean(text):
  text=text.lower()
  text=re.sub(r"[^a-zA-Z]"," ",text)
  return text

In [ ]:
#calling function
df['text']=df['text'].apply(clean)


In [ ]:
#convter from text to numrical bec nlp cant understand text
from sklearn.feature_extraction.text import TfidfVectorizer
convter=TfidfVectorizer(max_features=5000)


In [ ]:
#selecting x and y
x=convter.fit_transform(df['text'])
y=df['label']

# **train test split**

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=0)

# **model selction**

In [ ]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression()
model.fit(x_train,y_train)

LogisticRegression()

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.987305122494432

for saving files

In [ ]:
import pickle
pickle.dump(model,open('model.pkl','wb'))

In [ ]:
import gradio as gr
import pickle

# Load the trained model and TfidfVectorizer
model = pickle.load(open('model.pkl', 'rb'))
convter = TfidfVectorizer(max_features=5000) # Re-instantiate with same parameters

# To fit the convter, we need some data, so we will use the 'df' from the notebook state
# In a real deployment, you would save/load the fitted vectorizer or fit it on a representative dataset.
# For this demonstration, we'll refit it on the existing 'df['text']' to ensure it's in a working state.
# A more robust solution would be to pickle the convter object as well.
# If the 'convter' variable is directly available in the kernel, this step might be redundant, but it's safer.

# For the purpose of this Gradio interface, we need the fitted convter.
# If 'convter' was pickled, we would load it. Since it wasn't, we'll refit it using the original data.
# This assumes 'df' and the 'clean' function are still available and in the correct state.
# If 'df' is not available, this would require a re-run of data loading and cleaning steps.

# Re-fitting convter, assuming 'df' and 'clean' function are accessible
convter.fit(df['text'].apply(clean))

def predict_news(text):
    # Clean the input text
    cleaned_text = clean(text)
    # Transform the cleaned text using the loaded TfidfVectorizer
    vectorized_text = convter.transform([cleaned_text])
    # Make a prediction using the loaded model
    prediction = model.predict(vectorized_text)

    if prediction[0] == 0:
        return "Fake News"
    else:
        return "True News"

# Create the Gradio interface
interface = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=5, placeholder="Enter news article text here..."),
    outputs=gr.Label(),
    title="Fake News Detector",
    description="Enter a news article to determine if it's fake or true."
)

# Launch the interface
interface.launch(inline=True, share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>